# 2장 3강: MSE와 Cross Entropy의 수학적 원리 — 실습문제

## 실습 목표

- 실제값에서 예측값을 빼고, 제곱한 뒤 평균을 내어 MSE를 구합니다.
- 실제 구매 여부에 맞는 확률을 사용해 BCE를 계산합니다.
- 실제 상품 종류에 해당하는 확률을 골라 Categorical CE를 계산합니다.
- 주어진 Softmax 함수로 점수를 확률로 바꾸고 CE를 구합니다.
- 교안에서 MSE·BCE·CE가 NLL과 연결되는 뜻을 쉬운 설명으로 확인합니다.

## 실습 환경 / 데이터

Python과 NumPy를 사용합니다. 별도의 데이터 파일은 필요하지 않습니다. 데이터와 계산 방법은 제공된 「2장 3강: MSE와 Cross Entropy의 수학적 원리」의 본문 예제와 연습문제를 사용합니다.

회귀의 오차는 교안에 맞추어 **실제값 − 예측값**으로 계산합니다. 이진 분류는 **0=미구매, 1=구매**, 상품 분류는 **0=의류, 1=식품, 2=가전**입니다. 같은 숫자라도 문제에 따라 뜻이 다르므로 먼저 표의 제목을 확인하세요.

## 실습 구성과 교안 대응

| 구분 | 계산할 내용 | 교안 근거 |
|---|---|---|
| 필수 1 / 문제 1-1 | 두 금액 예측 모델의 오차·제곱오차·SSE·MSE | 1절 본문: 실제값 [10,20,30] 예제 |
| 필수 1 / 문제 1-2 | 고객별 BCE와 평균 BCE 비교 | 2절 본문: 고객 4명의 두 모델 비교 |
| 필수 2 / 문제 2-1 | 실제 정답 열 선택, 상품별 CE와 평균 CE | 3절 본문: 의류·식품·가전 상품 3개 예제 |
| 필수 2 / 문제 2-2 | 제공된 Softmax 함수로 확률과 CE 계산 | 3절 본문: Logit [2,1,0], [0,2,1], [1,0,3] |
| 과제 / 문제 3-1 | 새 금액 예측의 MSE 복습 | 연습문제 1의 MSE 계산 부분 |
| 과제 / 문제 3-2 | 새 고객 데이터의 BCE 복습 | 연습문제 2의 고객별·평균 BCE 계산 부분 |

**교안의 연습문제 전체를 그대로 옮긴 자료는 아닙니다.** 본문 예제와 연습문제의 일부를 기초 실습 4문제와 복습 과제 2문제로 재구성했습니다. 과제에서도 앞에서 익힌 계산식과 교안에 있는 숫자를 사용합니다.

## 진행 안내

1. NumPy를 불러오는 셀부터 실행합니다.
2. 표를 읽고 실제값·예측값 또는 실제 정답·예측 확률을 구분합니다.
3. 제곱은 각 오차에 먼저 적용하고, 그다음 평균을 구합니다.
4. 분류에서는 **실제 정답에 부여한 확률**을 찾은 다음 음의 로그를 적용합니다.
5. 제공된 함수와 확률 선택 코드는 그대로 사용해도 됩니다. 공식을 유도하거나 함수를 외워서 작성할 필요는 없습니다.
6. 계산 중간값은 반올림하지 않고 출력에만 `round()`·`np.round()`를 사용합니다.
7. 각 문제의 Q1~Q4에 계산값과 쉬운 문장으로 답하세요.

## 먼저 읽는 용어와 수식 기호

| 용어·기호 | 쉬운 뜻 |
|---|---|
| 손실 함수 / Loss Function | 모델의 예측을 정답과 비교하여 손실값으로 계산하는 규칙 |
| 회귀 / 분류 | 금액처럼 수치를 예측 / 정해진 종류를 예측 |
| y, y_true | 실제 정답. 금액인지 구매 여부인지 상품 종류인지 문제에 따라 확인 |
| ŷ, y_pred | 모델이 예측한 수치. ŷ는 ‘와이 햇’으로 읽음 |
| 오차 / errors | 실제값에서 예측값을 뺀 수 |
| 제곱 / ** 2 | 같은 수를 두 번 곱함. (−2)²=(−2)×(−2)=4 |
| SSE | 제곱오차를 모두 더한 값 |
| MSE | 제곱오차의 평균. SSE를 데이터 개수로 나눈 값 |
| p / 1−p | 이진 분류의 구매 확률 / 미구매 확률 |
| 클래스 / 인덱스 | 종류의 이름 / 배열에서의 위치 번호. 첫 위치는 0 |
| BCE | Binary Cross Entropy. 이 자료의 구매·미구매 분류에서 쓰는 손실 |
| CE | 이 자료에서는 Categorical Cross Entropy. 상품이 한 종류에만 속하는 분류의 손실 |
| One-Hot | 정답 클래스의 위치만 1, 나머지 위치는 0으로 표시한 배열 |
| Logit / Softmax | 확률로 바꾸기 전 점수 / 여러 점수를 합이 1인 확률로 바꾸는 함수 |
| n / Σ / ≈ | 데이터 개수 / 여러 값을 더하라는 기호 / 대략 같음 |
| ln / np.log | 자연로그 / 자연로그를 계산하는 NumPy 코드 |
| −ln(p) | 먼저 p의 자연로그를 구한 뒤 부호를 바꿈 |
| NLL | Negative Log-Likelihood. 우도의 로그에 마이너스를 붙인 값 |
| Mean / Total | 평균 / 합계 |

**로그를 처음 보는 경우:** 로그는 ‘어떤 수를 몇 제곱하면 주어진 값이 되는가’를 묻는 계산입니다. 자연로그는 약 2.718인 수 e를 기준으로 합니다. 이번 실습에서는 로그값을 손으로 구하지 않고 `np.log()`를 사용합니다. 예를 들어 `np.log(0.8)`은 약 −0.2231이고, `-np.log(0.8)`은 부호를 바꾼 약 0.2231입니다. 0과 1 사이 확률에 로그를 적용하면 음수이므로 앞의 마이너스를 빠뜨리지 마세요.

**손실 비교의 기준:** 같은 데이터·같은 손실 함수에서 값이 작은 쪽을 선택합니다. MSE와 BCE처럼 서로 다른 손실값을 숫자 크기만으로 비교하지 않습니다.

## 실습 준비

아래 셀을 먼저 실행하세요. NumPy는 숫자 배열을 계산하는 도구이고, `np`는 NumPy를 짧게 부르는 이름입니다.

In [ ]:
import numpy as np

---

## 과제. 필수 실습과 같은 방법으로 복습하기

### 문제 3-1. 새로운 금액 예측의 MSE 복습하기

#### 문제 설명

교안 연습문제 1의 MSE 계산을 복습합니다. 금액 단위는 만 원입니다.

| 고객 | 실제값 | 모델 A 예측 | 모델 B 예측 |
|---|---:|---:|---:|
| 첫째 | 8 | 10 | 8 |
| 둘째 | 12 | 11 | 16 |
| 셋째 | 20 | 18 | 17 |

문제 1-1과 같은 순서로 계산하세요. 새 공식은 필요하지 않습니다.

#### 요구사항

1. 실제값과 두 예측 배열을 만드세요.
2. 각 모델의 오차와 제곱오차를 계산해 출력하세요.
3. SSE와 MSE를 구하고 MSE는 소수점 아래 세 자리로 출력하세요.
4. Q1~Q4에 답하세요.

#### 작성 도움말

`오차 = 실제값 − 예측값`이며, `MSE = np.mean(오차 ** 2)`입니다. 모델 A의 첫 오차는 8−10입니다. 평균의 분모는 고객 수 3입니다.

#### 해석 질문

**Q1.** 두 모델의 오차와 제곱오차는 각각 무엇인가요?

**Q2.** SSE와 MSE는 각각 얼마이며 어떤 모델을 선택하나요?

**Q3.** 모델 B는 첫 고객을 정확히 예측했는데도 전체 MSE가 더 큰 이유는 무엇인가요?

**Q4.** 모델 B의 MSE를 계산할 때 오차가 있는 두 고객만 세어 2로 나누어도 되나요?

#### 제출 결과

- 요구사항의 계산 결과
- Q1~Q4 답변

In [ ]:
# 문제 3-1. 새로운 금액 예측의 MSE 복습하기
# 1. 문제에 제시된 배열과 필요한 함수를 작성하세요.
# 2. 요구사항 순서대로 계산하고 출력하세요.
# 3. 아래 답변란에 결과와 그 의미를 적으세요.

In [ ]:
y_true= np.array([8, 12, 20])
y_pred_A= np.array([10, 11, 18])
y_pred_B= np.array([8, 16, 17])

errors_A = y_true - y_pred_A
errors_B = y_true - y_pred_B
squared_errors_A = errors_A ** 2
squared_errors_B = errors_B ** 2

print("errors_A: \n", errors_A)
print("errors_B: \n", errors_B)
print("squared_errors_A: \n", squared_errors_A)
print("squared_errors_B: \n", squared_errors_B)

sse_A = np.sum(squared_errors_A)
sse_B = np.sum(squared_errors_B)
mse_A = np.mean(squared_errors_A)
mse_B = np.mean(squared_errors_B)
mse_A = np.round(mse_A, 3)
mse_B = np.round(mse_B, 3)
print("sse_A: \n", sse_A)
print("sse_B: \n", sse_B)
print("mse_A: \n", mse_A)
print("mse_B: \n", mse_B)

errors_A: 
 [-2  1  2]
errors_B: 
 [ 0 -4  3]
squared_errors_A: 
 [4 1 4]
squared_errors_B: 
 [ 0 16  9]
sse_A: 
 9
sse_B: 
 25
mse_A: 
 3.0
mse_B: 
 8.333


#### 답변 작성란

**Q1.** 두 모델의 오차와 제곱오차는 각각 무엇인가요?

답변:  
```python
errors_A: 
 [-2  1  2]
errors_B: 
 [ 0 -4  3]
squared_errors_A: 
 [4 1 4]
squared_errors_B: 
 [ 0 16  9]
 ```

**Q2.** SSE와 MSE는 각각 얼마이며 어떤 모델을 선택하나요?

답변: 
```python
sse_A: 
 9
sse_B: 
 25
mse_A: 
 3.0
mse_B: 
 8.333
 ```
 - 더 오차가 작은 A모델을 선택해야 한다.

**Q3.** 모델 B는 첫 고객을 정확히 예측했는데도 전체 MSE가 더 큰 이유는 무엇인가요?

답변: 다른 2개의 예측이 A모델보다 더 많이 빗나가있다. 
- MSE는 오차를 제곱하기 때문에 큰 오차를 가진 고객의 영향을 많이 받는다. 이것이 B의 MSE가 더 큰 이유이다.

**Q4.** 모델 B의 MSE를 계산할 때 오차가 있는 두 고객만 세어 2로 나누어도 되나요?

답변: MSE는 평균을 내는 것이기 때문에, 총 고객 수인 3으로 나누어야 한다.

### 문제 3-2. 새로운 고객 데이터의 BCE 복습하기

#### 문제 설명

교안 연습문제 2의 고객 3명입니다. 실제값은 1=구매, 0=미구매입니다. 표의 확률은 구매 확률입니다.

| 고객 | 실제값 y | 모델 A 구매 확률 | 모델 B 구매 확률 |
|---|---:|---:|---:|
| 첫째 | 1 | 0.8 | 0.6 |
| 둘째 | 0 | 0.3 | 0.6 |
| 셋째 | 1 | 0.9 | 0.7 |

문제 1-2에서 사용한 BCE 함수를 그대로 사용합니다.

#### 요구사항

1. 실제값과 두 모델의 확률 배열을 만드세요.
2. 아래 함수를 이용해 각 모델의 고객별 BCE를 계산하세요.
3. `np.mean()`으로 두 평균 BCE를 구하세요. 고객별·평균 BCE는 소수점 아래 세 자리로 출력합니다.
4. Q1~Q4에 답하세요.

#### 제공 함수와 작성 도움말

```python
def binary_cross_entropy(y_true, y_prob):
    return -(y_true * np.log(y_prob)
             + (1 - y_true) * np.log(1 - y_prob))
```

실제 정답이 0인 둘째 고객은 미구매 확률을 사용합니다. 모델 A에서는 1−0.3, 모델 B에서는 1−0.6입니다. 평균의 분모는 고객 수 3입니다.

#### 해석 질문

**Q1.** 두 모델이 실제 정답에 부여한 확률은 각각 무엇인가요?

**Q2.** 두 모델의 고객별 BCE와 평균 BCE는 각각 얼마인가요?

**Q3.** 모델 B가 첫째와 둘째에게 같은 구매 확률 0.6을 주었는데도 BCE가 다른 이유는 무엇인가요?

**Q4.** 평균 BCE 기준으로 어느 모델을 선택하며, 그 이유는 무엇인가요?

#### 제출 결과

- 요구사항의 계산 결과
- Q1~Q4 답변

In [ ]:
# 문제 3-2. 새로운 고객 데이터의 BCE 복습하기
# 1. 문제에 제시된 배열과 필요한 함수를 작성하세요.
# 2. 요구사항 순서대로 계산하고 출력하세요.
# 3. 아래 답변란에 결과와 그 의미를 적으세요.

In [ ]:
y_true= np.array([1, 0, 1])
y_prob_A= np.array([0.8, 0.3, 0.9])
y_prob_B= np.array([0.6, 0.6, 0.7])

print("y_prob_A: \n", y_prob_A)
print("y_prob_B: \n", y_prob_B)

def binary_cross_entropy(y_true, y_prob):
    return -(y_true * np.log(y_prob)
             + (1 - y_true) * np.log(1 - y_prob))


y_prob_A: 
 [0.8 0.3 0.9]
y_prob_B: 
 [0.6 0.6 0.7]


In [ ]:
BCE_A= binary_cross_entropy(y_true, y_prob_A)
BCE_B= binary_cross_entropy(y_true, y_prob_B)

print(f"BCE_A : \n{np.round(BCE_A, 3)}")
print(f"BCE_B : \n{np.round(BCE_B, 3)}")
print(f"모델 A 평균 BCE: {np.mean(BCE_A):.3f}")
print(f"모델 B 평균 BCE: {np.mean(BCE_B):.3f}")

BCE_A : 
[0.223 0.357 0.105]
BCE_B : 
[0.511 0.916 0.357]
모델 A 평균 BCE: 0.228
모델 B 평균 BCE: 0.595


#### 답변 작성란

**Q1.** 두 모델이 실제 정답에 부여한 확률은 각각 무엇인가요?

답변: 
```python
y_prob_A: 
 [0.8 0.3 0.9]
y_prob_B: 
 [0.6 0.6 0.7]
 ```
- A 모델이 부여한 확률: 0.8, 0.7, 0.9
- B 모델이 부여한 확률: 0.6, 0.4, 0.7

**Q2.** 두 모델의 고객별 BCE와 평균 BCE는 각각 얼마인가요?

답변: 
```python
BCE_A : 
[0.223 0.357 0.105]
BCE_B : 
[0.511 0.916 0.357]
모델 A 평균 BCE: 0.228
모델 B 평균 BCE: 0.595
```

**Q3.** 모델 B가 첫째와 둘째에게 같은 구매 확률 0.6을 주었는데도 BCE가 다른 이유는 무엇인가요?

답변: 첫 고객은 구매이고 둘 째 고객은 미구매이기 때문에, 모델 B는 둘째 고객에 대해 사실 상 0.4를 부여하게 된다.

**Q4.** 평균 BCE 기준으로 어느 모델을 선택하며, 그 이유는 무엇인가요?

답변: 평균 BCE 기준으로는 모델 A를 선택하게 된다. BCE 값이 작을수록 손실함수 값이 작아지기 때문이다.

---

## 실습 마무리

다음 질문으로 복습하세요. 6번은 교안의 연결 원리를 읽고 확인하는 질문입니다.

**1.** MSE를 구하는 계산 순서는 무엇인가요?

**2.** 실제값 10, 예측값 12일 때 오차 −2의 뜻은 무엇인가요?

**3.** 실제 미구매인데 구매 확률이 0.7이면 BCE에 어떤 확률을 쓰나요?

**4.** 상품 CE에서는 어떤 확률을 고르나요?

**5.** 상품별 CE를 구한 다음 평균은 무엇으로 나누나요?

**6.** MSE와 Gaussian NLL, BCE·CE와 NLL은 각각 어떻게 연결되나요?

### 실습 마무리 답변 작성란

**1.** 답변: 예측 값과 실제 값간의 오차를 구해서 제곱한 뒤 다 더하고, 데이터 수로 나눠서 평균을 구한다.

**2.** 답변: 예측 값에서 오차만큼(-2만큼) 가야 실제 값과 같아진다는 의미이다.

**3.** 답변: (1- p) 값인 0.3을 이용해 BCE를 구하게 된다.

**4.** 답변: 상품별로 실제 정답 클래스에 부여한 확률을 고릅니다. 가장 큰 예측 확률이 아닙니다. 예를 들어 정답이 가전(인덱스 2)이면 그 상품의 가전 확률을 골라 −ln(그 확률)로 CE를 계산합니다.

**5.** 답변: 총 상품 개수로 나눈다.

**6.** 답변: 세 가지 모두 **실제 관측 결과에 부여한 확률을 높이는 문제**, 즉 **NLL(Negative Log-Likelihood)을 최소화하는 문제**로 연결됩니다. Likelihood를 최대화하는 것은 NLL을 최소화하는 것과 같기 때문입니다.

| 손실 함수 | 확률 분포 가정 | 연결 관계 |
|---|---|---|
| MSE | 고정 분산의 정규분포(Gaussian) | `Mean NLL = 상수 + 양의 상수 × MSE` |
| BCE | 베르누이 분포 | 고객별 `−[y ln(p) + (1−y) ln(1−p)]`가 고객별 BCE이고, 이를 평균한 것이 평균 BCE |
| Categorical CE | 범주분포 | 상품별 `−ln(실제 정답 클래스의 확률)`이 상품별 CE이고, 이를 평균한 것이 평균 CE |

- **BCE ↔ 베르누이 NLL:** 구매/미구매 같은 이진 결과를 베르누이 분포로 보면, 고객별 NLL이 곧 고객별 BCE이고 이를 평균한 것이 평균 BCE(평균 NLL)입니다.
- **CE ↔ 범주분포 NLL:** 여러 클래스 중 하나가 정답인 경우, 실제 정답 클래스의 확률에 −ln을 적용한 값이 상품별 NLL이자 상품별 CE입니다. 이를 평균한 것이 평균 CE(평균 NLL)입니다.